In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from collections import Counter
import random
from typing import List, Dict

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# 1. Read data from attached files
# Assuming the files are in the current directory or provide paths
users = pd.read_csv('data_original/data_original/users.csv')
interactions = pd.read_csv('data_original/data_original/interactions.csv')
items = pd.read_csv('data_original/data_original/items.csv')

# Display basic info
print("Items DataFrame:")
print(items.info())
print(items.head())

print("\nInteractions DataFrame:")
print(interactions.info())
print(interactions.head())

print("\nUsers DataFrame:")
print(users.info())
print(users.head())

# Handle any encoding issues in items (titles seem garbled, assuming UTF-8 or cp1251)
# If titles are in Russian, you might need to reload with proper encoding, but since it's xlsx, pandas should handle it.
# For now, proceed.

# Convert dates in interactions
interactions['last_watch_dt'] = pd.to_datetime(interactions['last_watch_dt'])

# 2. Conduct EDA (Exploratory Data Analysis)

# 2.1 Basic statistics
print("\nItems Description:")
print(items.describe())

print("\nInteractions Description:")
print(interactions.describe())

print("\nUsers Description:")
print(users.describe())

# 2.2 Missing values
print("\nMissing values in Items:")
print(items.isnull().sum())

print("\nMissing values in Interactions:")
print(interactions.isnull().sum())

print("\nMissing values in Users:")
print(users.isnull().sum())

# 2.3 Distributions

# # Users age distribution
# plt.figure(figsize=(10, 5))
# sns.countplot(x='age', data=users)
# plt.title('User Age Distribution')
# plt.show()

# # Users income distribution
# plt.figure(figsize=(10, 5))
# sns.countplot(x='income', data=users)
# plt.title('User Income Distribution')
# plt.show()

# # Users sex distribution
# plt.figure(figsize=(10, 5))
# sns.countplot(x='sex', data=users)
# plt.title('User Sex Distribution')
# plt.show()

# # Interactions watched_pct distribution
# plt.figure(figsize=(10, 5))
# sns.histplot(interactions['watched_pct'], bins=20, kde=True)
# plt.title('Watched Percentage Distribution')
# plt.show()

# # Interactions total_dur distribution (log scale for better view)
# plt.figure(figsize=(10, 5))
# sns.histplot(np.log1p(interactions['total_dur']), bins=20, kde=True)
# plt.title('Log Total Duration Distribution')
# plt.show()

# # Number of interactions per user
# user_interactions = interactions.groupby('user_id').size().reset_index(name='count')
# plt.figure(figsize=(10, 5))
# sns.histplot(user_interactions['count'], bins=50, kde=True)
# plt.title('Interactions per User')
# plt.show()

# # Number of interactions per item
# item_interactions = interactions.groupby('item_id').size().reset_index(name='count')
# plt.figure(figsize=(10, 5))
# sns.histplot(item_interactions['count'], bins=50, kde=True)
# plt.title('Interactions per Item')
# plt.show()

# # Genres distribution (assuming genres are comma-separated)
# if 'genres' in items.columns:
#     genres_list = items['genres'].dropna().str.split(',').explode().str.strip()
#     plt.figure(figsize=(12, 6))
#     sns.countplot(y=genres_list, order=genres_list.value_counts().index[:20])
#     plt.title('Top 20 Genres')
#     plt.show()

# # Release year distribution
# if 'release_year' in items.columns:
#     plt.figure(figsize=(10, 5))
#     sns.histplot(items['release_year'].dropna(), bins=30, kde=True)
#     plt.title('Release Year Distribution')
#     plt.show()

# # 2.4 Correlations (for numerical features in interactions)
# corr = interactions[['total_dur', 'watched_pct']].corr()
# plt.figure(figsize=(8, 6))
# sns.heatmap(corr, annot=True, cmap='coolwarm')
# plt.title('Correlation Heatmap')
# plt.show()

# 3. Form dataset for training
# Merge interactions with users and items if needed, but for basic recsys, interactions suffice.
# For now, we'll use interactions primarily.

data = interactions.merge(users, on='user_id', how='left').merge(items, on='item_id', how='left')
print("\nMerged Data:")
print(data.head())

# 4. Split into train/test (test - last week)
max_date = interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=7)

train = interactions[interactions['last_watch_dt'] < test_start]
test = interactions[interactions['last_watch_dt'] >= test_start]

print(f"\nTrain shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Get unique users and items
all_users = interactions['user_id'].unique()
all_items = interactions['item_id'].unique()

# Group test by user for evaluation
test_user_items = test.groupby('user_id')['item_id'].apply(list).to_dict()

# Function to compute MAP@K
def average_precision(actual: List[int], predicted: List[int], k: int = 10) -> float:
    if not actual:
        return 0.0
    predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(actual), k)

def mean_average_precision(test_user_items: Dict[int, List[int]], recommendations: Dict[int, List[int]], k: int = 10) -> float:
    aps = []
    for user, actual in test_user_items.items():
        pred = recommendations.get(user, [])
        aps.append(average_precision(actual, pred, k))
    return np.mean(aps)

# 5. Make recommendations

# 5.1 Random
def random_recommender(train: pd.DataFrame, all_items: np.ndarray, k: int = 10) -> Dict[int, List[int]]:
    recommendations = {}
    for user in all_users:
        recommendations[user] = random.sample(list(all_items), k)
    return recommendations

random_recs = random_recommender(train, all_items)
map_random = mean_average_precision(test_user_items, random_recs)
print(f"MAP@10 for Random: {map_random}")

# 5.2 Popular (most interacted items)
def popular_recommender(train: pd.DataFrame, k: int = 10) -> Dict[int, List[int]]:
    popular_items = train['item_id'].value_counts().index[:k].tolist()
    recommendations = {user: popular_items for user in all_users}
    return recommendations

popular_recs = popular_recommender(train)
map_popular = mean_average_precision(test_user_items, popular_recs)
print(f"MAP@10 for Popular: {map_popular}")

# 5.3 TopPopular (personal popular + Popular)
# Assuming personal popular: top items watched by user in train, fill with global popular if less than k
def top_personal_popular(train: pd.DataFrame, popular_items: List[int], k: int = 10) -> Dict[int, List[int]]:
    user_items = train.groupby('user_id')['item_id'].apply(list).to_dict()
    recommendations = {}
    for user in all_users:
        personal = user_items.get(user, [])
        # Get unique top watched (but since it's interactions, assume most recent or count, here just unique)
        personal_counter = Counter(personal)
        top_personal = [item for item, _ in personal_counter.most_common(k)]
        if len(top_personal) < k:
            fill = [item for item in popular_items if item not in top_personal][:k - len(top_personal)]
            top_personal += fill
        recommendations[user] = top_personal
    return recommendations

popular_items = train['item_id'].value_counts().index.tolist()
top_popular_recs = top_personal_popular(train, popular_items)
map_top_popular = mean_average_precision(test_user_items, top_popular_recs)
print(f"MAP@10 for TopPopular: {map_top_popular}")

# 5.4 Weighted Popular
# Weighted by watched_pct or total_dur, here using mean watched_pct per item
def weighted_popular_recommender(train: pd.DataFrame, k: int = 10) -> Dict[int, List[int]]:
    item_weights = train.groupby('item_id')['watched_pct'].mean().sort_values(ascending=False)
    top_weighted = item_weights.index[:k].tolist()
    recommendations = {user: top_weighted for user in all_users}
    return recommendations

weighted_popular_recs = weighted_popular_recommender(train)
map_weighted = mean_average_precision(test_user_items, weighted_popular_recs)
print(f"MAP@10 for Weighted Popular: {map_weighted}")

Items DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15963 non-null  int64  
 1   content_type  15963 non-null  object 
 2   title         15963 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15865 non-null  float64
 5   genres        15963 non-null  object 
 6   countries     15926 non-null  object 
 7   for_kids      566 non-null    float64
 8   age_rating    15961 non-null  float64
 9   studios       1065 non-null   object 
 10  directors     14454 non-null  object 
 11  actors        13344 non-null  object 
 12  description   15961 non-null  object 
 13  keywords      15540 non-null  object 
dtypes: float64(3), int64(1), object(10)
memory usage: 1.7+ MB
None
   item_id content_type                 title      title_orig  release_year  \
0    10711         film        Поговори

In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta
from typing import List
from tqdm import tqdm

# ==============================================================================
## 1. Подготовка данных и генерация рекомендаций
# ==============================================================================

# Загрузка и подготовка данных
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'])

# Разделение на train/test
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=10)
df_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

# Определение уникальных пользователей, всех айтемов и истории просмотров
unique_test_users = df_test['user_id'].unique()
all_items = df_interactions['item_id'].unique()
user_history = df_train.groupby('user_id')['item_id'].apply(set).to_dict()

# Функция для генерации случайных рекомендаций
def generate_random_recommendations(user_id, all_items, history, k=10):
    seen_items = history.get(user_id, set())
    candidate_items = np.setdiff1d(all_items, list(seen_items), assume_unique=True)
    num_to_recommend = min(k, len(candidate_items))
    return np.random.choice(candidate_items, size=num_to_recommend, replace=False).tolist()

# Генерация случайных рекомендаций для всех тестовых пользователей
random_recs = {}
for user in tqdm(unique_test_users, desc="Generating Random Recommendations"):
    random_recs[user] = generate_random_recommendations(user, all_items, user_history, k=10)

# ==============================================================================
## 2. Подготовка фактических данных (Ground Truth)
# ==============================================================================

# Группируем реальные просмотры пользователей в тестовом периоде
ground_truth_df = df_test.groupby('user_id')['item_id'].apply(list).reset_index()

# Преобразуем в словарь для удобного использования в функции map_k
ground_truth_dict = pd.Series(ground_truth_df.item_id.values, index=ground_truth_df.user_id).to_dict()


# ==============================================================================
## 3. Функции для расчета MAP@10
# ==============================================================================

def ap_k(rec_list: List[int], true_list: List[int], k: int = 10) -> float:

    true_set = set(true_list)
    rec_list = rec_list[:k]
    
    if not true_set:
        return 0.0

    hits = 0
    score = 0.0
    for i, item in enumerate(rec_list, 1):
        if item in true_set:
            hits += 1
            score += hits / i
    
    return score / min(len(true_set), k)

def map_k(recommendations: dict, ground_truth: dict, k: int = 10) -> float:

    aps = []
    # Итерируемся по пользователям и их реальным просмотрам
    for user_id, true_list in ground_truth.items():
        # Находим рекомендации для этого пользователя
        rec_list = recommendations.get(user_id, [])
        if rec_list: # Считаем метрику, только если есть рекомендации
            aps.append(ap_k(rec_list, true_list, k))
            
    return np.mean(aps) if aps else 0.0


# ==============================================================================
## 4. Расчет и вывод результата
# ==============================================================================

map10_random = map_k(random_recs, ground_truth_dict, k=10)

print(f"\\nMAP@10 для случайных рекомендаций: {map10_random:.6f}")

Generating Random Recommendations: 100%|██████████| 230001/230001 [01:24<00:00, 2706.70it/s]


\nMAP@10 для случайных рекомендаций: 0.000222


In [ ]:
ground_truth = test_user_unique

In [ ]:
def ap_k(rec_list: List[int], true_list: List[int], k: int = 10) -> float:

    true_set = set(true_list)
    rec_list = rec_list[:k]
    
    if not true_set:
        return 0.0

    hits = 0
    score = 0.0
    for i, item in enumerate(rec_list, 1):
        if item in true_set:
            hits += 1
            score += hits / i
    
    return score / min(len(true_set), k)

In [ ]:
def map_k(recommendations: dict, ground_truth: dict, k: int = 10) -> float:

    aps = []
    # Итерируемся по пользователям, для которых у нас есть рекомендации
    for user_id, rec_list in recommendations.items():
        # Находим реальные просмотры этого пользователя из ground_truth
        true_list = ground_truth.get(user_id, [])
        if true_list: # Считаем метрику только если есть что-то для сравнения
            aps.append(ap_k(rec_list, true_list, k))
            
    return np.mean(aps)

In [ ]:
def recommend_random(user_id, user_history, k = 10) -> List[int]:
    list = user_history.get(user_id, set())
    rec_items = list(user_history.keys())
    k = min(k, len(rec_items))
    return np.random.choice(rec_items, k, replace=False).tolist()


In [ ]:
map_random = map_k(random_recs, ground_truth, k=10)
map_random